<a href="https://colab.research.google.com/github/Quijano89/Trial1_codekada/blob/main/Codekada_final_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------
# USER INPUT: MASS (STRICT)
# -------------------------
m = float(input("Enter mass (kg): "))

if m <= 0:
    raise ValueError("Mass must be positive. No assumptions allowed.")

In [ ]:
import cv2
import numpy as np
import os

# -------------------------
# VIDEO INPUT (USER)
# -------------------------
video_path = input("Enter video path: ").strip()

print("Working dir:", os.getcwd())

if not os.path.exists(video_path):
    raise FileNotFoundError("Video not found.")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video failed to open.")

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read first frame.")

# -------------------------
# DISPLAY FIX
# -------------------------
display_scale = 0.6
cv2.namedWindow("Stable Physics Tracking", cv2.WINDOW_NORMAL)

# -------------------------
# TRACKER
# -------------------------
def create_tracker():
    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()
    elif hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()
    else:
        return cv2.TrackerKCF_create()

# -------------------------
# ROI SELECTION
# -------------------------
roi_scale = 0.5
frame_small = cv2.resize(frame, (0, 0), fx=roi_scale, fy=roi_scale)

bbox_small = cv2.selectROI("Select Object", frame_small, False)
cv2.destroyAllWindows()

x, y, w, h = bbox_small
bbox = (int(x / roi_scale), int(y / roi_scale), int(w / roi_scale), int(h / roi_scale))

tracker = create_tracker()
tracker.init(frame, bbox)

# -------------------------
# STATE
# -------------------------
centers = []
trail = []

lost_frames = 0
max_lost = 25

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

smooth_cx, smooth_cy = None, None
vx, vy = 0.0, 0.0

alpha = 0.35
beta = 0.75

# -------------------------
# MAIN LOOP
# -------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        lost_frames = 0

        x, y, w, h = map(int, bbox)

        roi = frame[y:y+h, x:x+w]
        if roi.size == 0:
            continue

        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        _, mask = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        M = cv2.moments(mask)

        if M["m00"] != 0:
            dx = M["m10"] / M["m00"]
            dy = M["m01"] / M["m00"]
        else:
            dx, dy = w / 2, h / 2

        cx = x + dx
        cy = y + dy

        if len(centers) > 2:
            px, py = centers[-1]
            if abs(cx - px) > 150 or abs(cy - py) > 150:
                continue

        if smooth_cx is None:
            smooth_cx, smooth_cy = cx, cy
        else:
            vx = beta * vx + (1 - beta) * (cx - smooth_cx)
            vy = beta * vy + (1 - beta) * (cy - smooth_cy)

            smooth_cx += vx
            smooth_cy += vy

            smooth_cx = alpha * cx + (1 - alpha) * smooth_cx
            smooth_cy = alpha * cy + (1 - alpha) * smooth_cy

        centers.append((smooth_cx, smooth_cy))
        trail.append((int(smooth_cx), int(smooth_cy)))

        if len(trail) > 500:
            trail = trail[-500:]

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.circle(frame, (int(smooth_cx), int(smooth_cy)), 4, (0, 0, 255), -1)

        start = max(1, len(trail) - 120)
        for i in range(start, len(trail)):
            cv2.line(frame, trail[i - 1], trail[i], (255, 0, 0), 2)

    else:
        lost_frames += 1

        cv2.putText(
            frame,
            f"Lost ({lost_frames}/{max_lost})",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

        if lost_frames > max_lost:
            bbox_small = cv2.selectROI("Re-select object", frame, False)
            cv2.destroyAllWindows()

            x, y, w, h = bbox_small
            bbox = (x, y, w, h)

            tracker = create_tracker()
            tracker.init(frame, bbox)

            lost_frames = 0
            trail.clear()
            smooth_cx, smooth_cy = None, None
            vx, vy = 0.0, 0.0

    frame_display = cv2.resize(frame, (0, 0), fx=display_scale, fy=display_scale)
    cv2.imshow("Stable Physics Tracking", frame_display)

    wait_time = max(1, int(1000 / fps))
    if cv2.waitKey(wait_time) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# -------------------------
# OUTPUT (NO CSV)
# -------------------------
centers = np.array(centers, dtype=float)

print("\nCV COMPLETE")
print("samples:", len(centers))
print("fps:", fps)

# THIS is what you pass to physics code:
cv_data = {
    "centers": centers,
    "fps": fps
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

plt.style.use("seaborn-v0_8-darkgrid")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
    "legend.frameon": True,
})

# =========================================================
# INPUT FROM CV (NO CSV)
# =========================================================

centers = cv_data["centers"]
fps = cv_data["fps"]

m = cv_data.get("m", None)
if m is None:
    m = float(input("Enter mass (kg): "))
    if m <= 0:
        raise ValueError("Mass must be positive.")

centers = np.array(centers)

# =========================================================
# PHYSICS PIPELINE
# =========================================================

t = np.arange(len(centers)) / fps
y = -centers[:, 1]

window = min(11, len(y)//2*2 - 1)
window = max(window, 5)
y_smooth = savgol_filter(y, window, 3)

y0 = y_smooth - np.mean(y_smooth)
trend = np.polyfit(t, y0, 1)
y0 = y0 - (trend[0] * t + trend[1])

v = np.gradient(y0, 1/fps)

# =========================================================
# FREQUENCY (ω)
# =========================================================

fft = np.fft.fft(y0)
freqs = np.fft.fftfreq(len(y0), d=1/fps)

mask = freqs > 0
dominant_freq = freqs[mask][np.argmax(np.abs(fft[mask]))]
omega = 2 * np.pi * dominant_freq

# =========================================================
# DAMPING (γ)
# =========================================================

peaks, _ = find_peaks(y0, distance=10)
peaks_t = t[peaks]
peaks_y = np.abs(y0[peaks])

valid = peaks_y > 0.1 * np.max(peaks_y)
peaks_t = peaks_t[valid]
peaks_y = peaks_y[valid]

gamma = -np.polyfit(peaks_t, np.log(peaks_y + 1e-12), 1)[0]

zeta = gamma / np.sqrt(omega**2 + gamma**2)

# =========================================================
# 🔥 NEW: NATURAL FREQUENCY (UNDAMPED)
# =========================================================

omega0 = np.sqrt(omega**2 + gamma**2)

# =========================================================
# 🔥 NEW: REGIME CLASSIFICATION
# =========================================================

if zeta < 1:
    regime = "Underdamped"
elif np.isclose(zeta, 1, atol=0.05):
    regime = "Critically damped"
else:
    regime = "Overdamped"

# =========================================================
# SPRING CONSTANT
# =========================================================

k = m * omega0**2

# =========================================================
# 🔥 NEW: PHASE ESTIMATION (IMPORTANT FIX)
# =========================================================

# remove damping envelope first
y_damped_removed = y0 * np.exp(gamma * t)

A = np.column_stack([np.cos(omega * t), np.sin(omega * t)])
coef, _, _, _ = np.linalg.lstsq(A, y_damped_removed, rcond=None)

B, C = coef

A_amp = np.sqrt(B**2 + C**2)
phi = np.arctan2(-C, B)

# corrected model
y_model = np.exp(-gamma * t) * A_amp * np.cos(omega * t + phi)

# =========================================================
# ENERGY
# =========================================================

E = 0.5 * m * v**2 + 0.5 * k * y0**2

# =========================================================
# FIT ERROR
# =========================================================

rmse = np.sqrt(np.mean((y0 - y_model)**2))
nrmse = rmse / (np.max(y0) - np.min(y0))

# =========================================================
# STABILITY CHECK
# =========================================================

mid = len(y0)//2

def estimate(x, tseg):
    fft = np.fft.fft(x)
    f = np.fft.fftfreq(len(x), 1/fps)
    mask = f > 0
    w = 2*np.pi*f[mask][np.argmax(np.abs(fft[mask]))]

    p, _ = find_peaks(x, distance=10)
    pt = tseg[p]
    py = np.abs(x[p])

    g = -np.polyfit(pt, np.log(py + 1e-12), 1)[0]
    return w, g

w1, g1 = estimate(y0[:mid], t[:mid])
w2, g2 = estimate(y0[mid:], t[mid:])

# =========================================================
# OUTPUT
# =========================================================

print("\nRESULTS")
print("omega:", omega)
print("omega0:", omega0)
print("gamma:", gamma)
print("k:", k)
print("zeta:", zeta)
print("phase phi:", phi)

print("\nREGIME")
print(regime)

print("\nSTABILITY")
print("omega drift %:", abs(w1-w2)/w2*100)
print("gamma drift %:", abs(g1-g2)/g2*100)

print("\nFIT")
print("RMSE:", rmse)
print("NRMSE:", nrmse)

# =========================================================
# PLOTS
# =========================================================

plt.figure()
plt.plot(t, y0, label="data")
plt.plot(t, y_model, "--", label="model")
plt.title("Spring Motion Fit")
plt.legend()
plt.show()

plt.figure()
plt.plot(t, E, color="purple")
plt.title("Mechanical Energy")
plt.show()

plt.figure()
plt.plot(y0, v, color="black")
plt.title("Phase Space")
plt.xlabel("x")
plt.ylabel("v")
plt.show()

plt.figure()
plt.scatter(peaks_t, peaks_y)
env = np.exp(np.polyval(np.polyfit(peaks_t, np.log(peaks_y+1e-12), 1), peaks_t))
plt.plot(peaks_t, env)
plt.title("Damping Envelope")
plt.show()